##### NOTEBOOK 3 — CAPA GOLD
##### TFM: Integración QML en pipeline DataOps
##### Juan Albornoz Carrasco — Universidad Europea de Valencia
##### Lee desde Silver Parquet — Escribe Gold Delta Lake + Parquet


##### CELDA 1 — Imports y configuracion


In [0]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [0]:
spark = SparkSession.builder.getOrCreate()

pd.set_option("display.max_columns", None)

silver_dir = "/Volumes/workspace/default/nhanes/silver"
gold_path  = "/Volumes/workspace/default/nhanes/gold_delta"
gold_dir   = "/Volumes/workspace/default/nhanes/gold"
models_dir = "/Volumes/workspace/default/nhanes/models"

os.makedirs(gold_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
print("Imports OK")
print(f"Silver Parquet: {silver_dir}")
print(f"Gold Delta:     {gold_path}")
print(f"Gold Parquet:   {gold_dir}")

Imports OK
Silver Parquet: /Volumes/workspace/default/nhanes/silver
Gold Delta:     /Volumes/workspace/default/nhanes/gold_delta
Gold Parquet:   /Volumes/workspace/default/nhanes/gold


##### CELDA 2 — Cargar datasets Silver desde Parquet


In [0]:
df_lgbm = pd.read_parquet(f"{silver_dir}/silver_lgbm.parquet")
df_svm  = pd.read_parquet(f"{silver_dir}/silver_svm.parquet")

print(f"Silver LightGBM: {df_lgbm.shape}")
print(f"Silver SVM/QSVM: {df_svm.shape}")

Silver LightGBM: (7831, 91)
Silver SVM/QSVM: (7831, 91)


##### CELDA 3 — Definir columnas a excluir del modelado
##### Excluir leakage conceptual, identificadores y admin


In [0]:
cols_excluir = [
    "SEQN", "TARGET", "DIQ010",
    "DIQ040", "DIQ070", "DIQ160",
    "DIQ170", "DIQ172", "DIQ180",
    "DIQ050", "CICLO", "WTSAF2YR",
    "SDDSRVYR", "RIDSTATR",
]

cols_excluir = [c for c in cols_excluir if c in df_svm.columns]
feature_cols = [c for c in df_svm.columns if c not in cols_excluir]
print(f"Columnas excluidas: {cols_excluir}")
print(f"Features disponibles: {len(feature_cols)}")

Columnas excluidas: ['SEQN', 'TARGET', 'DIQ010', 'CICLO', 'WTSAF2YR', 'SDDSRVYR', 'RIDSTATR']
Features disponibles: 84


##### CELDA 4 — One-hot encoding variables categoricas


In [0]:
vars_categoricas = ["RIAGENDR", "RIDRETH1", "RIDRETH3", "DMDEDUC2", "DMDMARTL"]
vars_categoricas_presentes = [v for v in vars_categoricas if v in feature_cols]
print(f"Variables categoricas para encoding: {vars_categoricas_presentes}")

df_svm_encoded = pd.get_dummies(
    df_svm[feature_cols + ["TARGET"]],
    columns=vars_categoricas_presentes,
    drop_first=False
)
df_lgbm_encoded = pd.get_dummies(
    df_lgbm[feature_cols + ["TARGET"]],
    columns=vars_categoricas_presentes,
    drop_first=False
)

print(f"\nPost encoding SVM:   {df_svm_encoded.shape}")
print(f"Post encoding LGBM:  {df_lgbm_encoded.shape}")

Variables categoricas para encoding: ['RIAGENDR', 'RIDRETH1', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTL']

Post encoding SVM:   (7831, 106)
Post encoding LGBM:  (7831, 106)


##### CELDA 5 — Eliminar variables con alta correlacion (r > 0.90)


In [0]:
X_svm = df_svm_encoded.drop(columns=["TARGET"])
corr_matrix = X_svm.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
cols_alta_corr = [col for col in upper.columns if any(upper[col] > 0.90)]

print(f"Variables con correlacion > 0.90: {len(cols_alta_corr)}")
print(cols_alta_corr)

df_svm_encoded  = df_svm_encoded.drop(columns=cols_alta_corr)
df_lgbm_encoded = df_lgbm_encoded.drop(columns=cols_alta_corr, errors="ignore")

print(f"\nPost eliminacion correlacion SVM:  {df_svm_encoded.shape}")
print(f"Post eliminacion correlacion LGBM: {df_lgbm_encoded.shape}")

Variables con correlacion > 0.90: 16
['DMDFMSIZ', 'WTMEC2YR', 'INDFMIN2', 'WTSAF2YR_x', 'LBDGLUSI', 'WTSAF2YR_y', 'LBDINSI', 'BPXSY2', 'BPXSY3', 'LBDTRSI', 'LBDLDLSI', 'RIAGENDR_2.0', 'RIDRETH3_1.0', 'RIDRETH3_2.0', 'RIDRETH3_3.0', 'RIDRETH3_4.0']

Post eliminacion correlacion SVM:  (7831, 90)
Post eliminacion correlacion LGBM: (7831, 90)


##### CELDA 6 — Particion train/test 80/20 estratificada


In [0]:
X_svm  = df_svm_encoded.drop(columns=["TARGET"])
y_svm  = df_svm_encoded["TARGET"]
X_lgbm = df_lgbm_encoded.drop(columns=["TARGET"])
y_lgbm = df_lgbm_encoded["TARGET"]

X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(
    X_svm, y_svm, test_size=0.20, random_state=42, stratify=y_svm
)
X_train_lgbm, X_test_lgbm, y_train_lgbm, y_test_lgbm = train_test_split(
    X_lgbm, y_lgbm, test_size=0.20, random_state=42, stratify=y_lgbm
)

print("=== PARTICION TRAIN/TEST ===")
print(f"SVM  — Train: {X_train_svm.shape}, Test: {X_test_svm.shape}")
print(f"LGBM — Train: {X_train_lgbm.shape}, Test: {X_test_lgbm.shape}")
print(f"\nDistribucion target en train SVM:")
print(y_train_svm.value_counts(normalize=True).mul(100).round(2))
print(f"\nDistribucion target en test SVM:")
print(y_test_svm.value_counts(normalize=True).mul(100).round(2))

=== PARTICION TRAIN/TEST ===
SVM  — Train: (6264, 89), Test: (1567, 89)
LGBM — Train: (6264, 89), Test: (1567, 89)

Distribucion target en train SVM:
0    85.97
1    14.03
Name: TARGET, dtype: float64

Distribucion target en test SVM:
0    85.96
1    14.04
Name: TARGET, dtype: float64


##### CELDA 7 — Normalizacion StandardScaler
##### Ajuste solo sobre train — evitar data leakage
##### Solo para SVM y QSVM


In [0]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_svm_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_svm),
    columns=X_train_svm.columns
)
X_test_svm_scaled = pd.DataFrame(
    scaler.transform(X_test_svm),
    columns=X_test_svm.columns
)

print(f"Train SVM escalado: {X_train_svm_scaled.shape}")
print(f"Test SVM escalado:  {X_test_svm_scaled.shape}")

Train SVM escalado: (6264, 89)
Test SVM escalado:  (1567, 89)


In [0]:
# Verificacion escala — excluir columnas con varianza cero
cols_validas = [c for c in X_train_svm_scaled.columns 
                if X_train_svm_scaled[c].std() > 0.01]

stats_val = X_train_svm_scaled[cols_validas].agg(['mean', 'std']).T
media_ok  = (stats_val['mean'].abs() < 0.01).all()
std_ok    = (stats_val['std'].between(0.90, 1.10)).all()

print(f"\nVerificacion escala (media ~0, std ~1):")
print(f"  Columnas evaluadas (varianza > 0): {len(cols_validas)}")
print(f"  Columnas excluidas (varianza = 0): {89 - len(cols_validas)}")
print(f"  Media ~0: {media_ok}")
print(f"  Std  ~1:  {std_ok}")
if media_ok and std_ok:
    print(">>> StandardScaler aplicado correctamente")
else:
    print(">>> ⚠️ Revisar escalado")


Verificacion escala (media ~0, std ~1):
  Columnas evaluadas (varianza > 0): 66
  Columnas excluidas (varianza = 0): 23
  Media ~0: True
  Std  ~1:  True
>>> StandardScaler aplicado correctamente


##### CELDA 8 — Seleccion top 8 features para QSVM
##### Random Forest sobre features sin variables DIQ (leakage)

In [0]:
vars_excluir_qsvm = ["DIQ070", "DIQ172", "DIQ180", "DIQ040", "DIQ160", "DIQ170"]
features_sin_diq = [
    c for c in X_train_svm_scaled.columns
    if not any(c.startswith(excl) for excl in vars_excluir_qsvm)
    and not c.startswith("DIQ")
]
print(f"Features disponibles sin variables DIQ: {len(features_sin_diq)}")

X_train_nodiq = X_train_svm_scaled[features_sin_diq]
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_nodiq, y_train_svm)

importancias = pd.Series(
    rf.feature_importances_,
    index=features_sin_diq
).sort_values(ascending=False)

top8_features = importancias.head(8).index.tolist()
print("\nTop 8 features para QSVM (sin variables DIQ):")
for i, feat in enumerate(top8_features, 1):
    print(f"  {i}. {feat}: {importancias[feat]:.4f}")

Features disponibles sin variables DIQ: 89

Top 8 features para QSVM (sin variables DIQ):
  1. LBXGH: 0.2452
  2. LBXGLU: 0.1853
  3. RIDAGEYR: 0.0325
  4. LBDLDL: 0.0315
  5. BMXWAIST: 0.0284
  6. LBXIN: 0.0264
  7. BMXLEG: 0.0226
  8. WTINT2YR: 0.0221


##### CELDA 9 — Muestra estratificada para QSVM

In [0]:
X_train_qsvm, _, y_train_qsvm, _ = train_test_split(
    X_train_svm_scaled,
    y_train_svm,
    train_size=1500,
    random_state=42,
    stratify=y_train_svm
)

X_train_qsvm_8 = X_train_qsvm[top8_features]
X_test_qsvm_8  = X_test_svm_scaled[top8_features]

print(f"Muestra QSVM train: {X_train_qsvm.shape}")
print(f"QSVM train reducido: {X_train_qsvm_8.shape}")
print(f"QSVM test reducido:  {X_test_qsvm_8.shape}")
print(f"Distribucion target QSVM train:")
print(y_train_qsvm.value_counts(normalize=True).mul(100).round(2))

Muestra QSVM train: (1500, 89)
QSVM train reducido: (1500, 8)
QSVM test reducido:  (1567, 8)
Distribucion target QSVM train:
0    86.0
1    14.0
Name: TARGET, dtype: float64


##### CELDA 10 — Resumen final Gold

In [0]:
print("=== RESUMEN CAPA GOLD ===")
print(f"\nDataset LightGBM:")
print(f"  Train: {X_train_lgbm.shape[0]} registros, {X_train_lgbm.shape[1]} features")
print(f"  Test:  {X_test_lgbm.shape[0]} registros, {X_test_lgbm.shape[1]} features")
print(f"\nDataset SVM-RBF:")
print(f"  Train: {X_train_svm_scaled.shape[0]} registros, {X_train_svm_scaled.shape[1]} features")
print(f"  Test:  {X_test_svm_scaled.shape[0]} registros, {X_test_svm_scaled.shape[1]} features")
print(f"\nDataset QSVM (muestra estratificada):")
print(f"  Train: {X_train_qsvm_8.shape[0]} registros, {X_train_qsvm_8.shape[1]} features")
print(f"  Test:  {X_test_qsvm_8.shape[0]} registros, {X_test_qsvm_8.shape[1]} features")

=== RESUMEN CAPA GOLD ===

Dataset LightGBM:
  Train: 6264 registros, 89 features
  Test:  1567 registros, 89 features

Dataset SVM-RBF:
  Train: 6264 registros, 89 features
  Test:  1567 registros, 89 features

Dataset QSVM (muestra estratificada):
  Train: 1500 registros, 8 features
  Test:  1567 registros, 8 features


##### CELDA 11 — Guardar Gold como Parquet
##### Compatibilidad con notebooks LightGBM, SVM, QSVM


In [0]:
# LightGBM
X_train_lgbm.to_parquet(f"{gold_dir}/X_train_lgbm.parquet", index=False)
X_test_lgbm.to_parquet(f"{gold_dir}/X_test_lgbm.parquet", index=False)
y_train_lgbm.to_frame().to_parquet(f"{gold_dir}/y_train_lgbm.parquet", index=False)
y_test_lgbm.to_frame().to_parquet(f"{gold_dir}/y_test_lgbm.parquet", index=False)

# SVM-RBF
X_train_svm_scaled.to_parquet(f"{gold_dir}/X_train_svm.parquet", index=False)
X_test_svm_scaled.to_parquet(f"{gold_dir}/X_test_svm.parquet", index=False)
y_train_svm.to_frame().to_parquet(f"{gold_dir}/y_train_svm.parquet", index=False)
y_test_svm.to_frame().to_parquet(f"{gold_dir}/y_test_svm.parquet", index=False)

# QSVM
X_train_qsvm_8.to_parquet(f"{gold_dir}/X_train_qsvm_8features.parquet", index=False)
X_test_qsvm_8.to_parquet(f"{gold_dir}/X_test_qsvm_8features.parquet", index=False)
y_train_qsvm.to_frame().to_parquet(f"{gold_dir}/y_train_qsvm.parquet", index=False)
y_test_svm.to_frame().to_parquet(f"{gold_dir}/y_test_qsvm.parquet", index=False)
pd.Series(top8_features).to_csv(f"{gold_dir}/qsvm_top8_features.csv", index=False)

# Scaler
joblib.dump(scaler, f"{models_dir}/scaler.pkl")

print("Datasets Gold Parquet guardados:")
print(f"  LightGBM: X_train/X_test/y_train/y_test")
print(f"  SVM-RBF:  X_train/X_test/y_train/y_test")
print(f"  QSVM:     X_train_8/X_test_8/y_train + top8_features.csv")
print(f"  Scaler:   {models_dir}/scaler.pkl")

Datasets Gold Parquet guardados:
  LightGBM: X_train/X_test/y_train/y_test
  SVM-RBF:  X_train/X_test/y_train/y_test
  QSVM:     X_train_8/X_test_8/y_train + top8_features.csv
  Scaler:   /Volumes/workspace/default/nhanes/models/scaler.pkl


##### CELDA 12 — Materializar Gold en Delta Lake
##### Dataset Gold SVM usado como Gold canonico


In [0]:
# Construcción del dataset Gold completo SVM
df_gold_delta = X_train_svm_scaled.copy()
df_gold_delta["TARGET"] = y_train_svm.values
df_gold_delta["SPLIT"] = "train"

df_gold_test = X_test_svm_scaled.copy()
df_gold_test["TARGET"] = y_test_svm.values
df_gold_test["SPLIT"] = "test"

df_gold_full = pd.concat([df_gold_delta, df_gold_test], axis=0).reset_index(drop=True)

df_gold_spark = spark.createDataFrame(df_gold_full)

df_gold_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

print(f"Gold materializado en Delta Lake")
print(f"Registros: {df_gold_spark.count()}")
print(f"Columnas:  {len(df_gold_spark.columns)}")
print(f"Ruta:      {gold_path}")

Gold materializado en Delta Lake
Registros: 7831
Columnas:  91
Ruta:      /Volumes/workspace/default/nhanes/gold_delta


##### CELDA 13 — Verificacion Delta Lake Gold


In [0]:
delta_table = DeltaTable.forPath(spark, gold_path)
historial = delta_table.history().toPandas()
cols_principales = ["version", "timestamp", "operation", "operationMetrics"]
display(historial[cols_principales])

version,timestamp,operation,operationMetrics
7,2026-06-23T20:13:23.000Z,WRITE,"List(0, 8, 755326, 7831, 755326, 8)"
6,2026-06-23T20:11:48.000Z,WRITE,"List(0, 8, 755326, 7831, 754078, 8)"
5,2026-06-21T06:13:12.000Z,WRITE,"List(0, 8, 754078, 7831, 754078, 8)"
4,2026-06-21T06:04:25.000Z,WRITE,"List(0, 8, 754078, 7831, 754078, 8)"
3,2026-06-20T22:09:51.000Z,WRITE,"List(0, 8, 754078, 7831, 754230, 8)"
2,2026-06-07T16:49:10.000Z,WRITE,"List(0, 8, 754230, 7831, 756228, 8)"
1,2026-06-01T19:47:11.000Z,WRITE,"List(0, 8, 756228, 7831, 756228, 8)"
0,2026-05-31T17:50:46.000Z,WRITE,"List(0, 8, 756228, 7831, 0, 0)"


##### CELDA 14 — Validacion final Gold

In [0]:
print("=== VALIDACION FINAL CAPA GOLD ===")

print("\n1. Parquet guardados:")
for f in os.listdir(gold_dir):
    path = f"{gold_dir}/{f}"
    if f.endswith(".parquet"):
        df_check = pd.read_parquet(path)
        print(f"   {f}: {df_check.shape}")

print("\n2. Delta Lake:")
df_verify = spark.read.format("delta").load(gold_path)
print(f"   Registros: {df_verify.count()}")
print(f"   Columnas:  {len(df_verify.columns)}")
print(f"   Splits:")
df_verify.groupBy("SPLIT").count().show()

print("\n3. Archivos en Unity Catalog Volumes:")
files = dbutils.fs.ls("/Volumes/workspace/default/nhanes/gold/")
for f in files:
    print(f"   {f.name}")

print("\n=== GOLD VALIDADO OK ===")

=== VALIDACION FINAL CAPA GOLD ===

1. Parquet guardados:
   X_train_lgbm.parquet: (6264, 89)
   X_test_lgbm.parquet: (1567, 89)
   y_train_lgbm.parquet: (6264, 1)
   y_test_lgbm.parquet: (1567, 1)
   X_train_svm.parquet: (6264, 89)
   X_test_svm.parquet: (1567, 89)
   y_train_svm.parquet: (6264, 1)
   y_test_svm.parquet: (1567, 1)
   X_train_qsvm_8features.parquet: (1500, 8)
   X_test_qsvm_8features.parquet: (1567, 8)
   y_train_qsvm.parquet: (1500, 1)
   y_test_qsvm.parquet: (1567, 1)
   X_train_qsvm.parquet: (1500, 89)

2. Delta Lake:
   Registros: 7831
   Columnas:  91
   Splits:
+-----+-----+
|SPLIT|count|
+-----+-----+
| test| 1567|
|train| 6264|
+-----+-----+


3. Archivos en Unity Catalog Volumes:
   X_test_lgbm.parquet
   X_test_qsvm_8features.parquet
   X_test_svm.parquet
   X_train_lgbm.parquet
   X_train_qsvm.parquet
   X_train_qsvm_8features.parquet
   X_train_svm.parquet
   qsvm_top8_features.csv
   y_test_lgbm.parquet
   y_test_qsvm.parquet
   y_test_svm.parquet
   y_tra

##### CELDA 15 — Time Travel Delta Lake: consulta versión histórica


In [0]:
gold_path = "/Volumes/workspace/default/nhanes/gold_delta"

# Historial de versiones disponibles
delta_table = DeltaTable.forPath(spark, gold_path)
historial   = delta_table.history().toPandas()

print("=== TIME TRAVEL — HISTORIAL GOLD ===\n")
print(historial[["version", "timestamp", "operation", "operationMetrics"]].to_string(index=False))

# Leer versión anterior disponible (version 1)
df_v1     = spark.read.format("delta").option("versionAsOf", 7).load(gold_path)
df_actual = spark.read.format("delta").load(gold_path)

print(f"\nVersión 1 (2026-06-01): {df_v1.count()} registros, {len(df_v1.columns)} columnas")
print(f"Versión actual (v7):    {df_actual.count()} registros, {len(df_actual.columns)} columnas")
print("\n✅ Time travel verificado correctamente")

=== TIME TRAVEL — HISTORIAL GOLD ===

 version           timestamp operation                                                                                                                                              operationMetrics
       7 2026-06-23 20:13:23     WRITE {'numFiles': '8', 'numRemovedFiles': '8', 'numRemovedBytes': '755326', 'numDeletionVectorsRemoved': '0', 'numOutputRows': '7831', 'numOutputBytes': '755326'}
       6 2026-06-23 20:11:48     WRITE {'numFiles': '8', 'numRemovedFiles': '8', 'numRemovedBytes': '754078', 'numDeletionVectorsRemoved': '0', 'numOutputRows': '7831', 'numOutputBytes': '755326'}
       5 2026-06-21 06:13:12     WRITE {'numFiles': '8', 'numRemovedFiles': '8', 'numRemovedBytes': '754078', 'numDeletionVectorsRemoved': '0', 'numOutputRows': '7831', 'numOutputBytes': '754078'}
       4 2026-06-21 06:04:25     WRITE {'numFiles': '8', 'numRemovedFiles': '8', 'numRemovedBytes': '754078', 'numDeletionVectorsRemoved': '0', 'numOutputRows': '7831', 'numO